# Stock Market Prediction using LSTM
### Deep Learning pipeline: Data Cleaning -> Feature Engineering -> LSTM Model -> Best Accuracy

Dataset: Apple (AAPL), Google (GOOG), Microsoft (MSFT), Amazon (AMZN) via yfinance
Model: Stacked Bidirectional LSTM with Dropout + Early Stopping

Steps:
1. Install & Import Libraries
2. Download & Inspect Data
3. Data Cleaning & Preprocessing
4. Exploratory Data Analysis (EDA)
5. Feature Engineering (Technical Indicators)
6. Build & Train LSTM Model
7. Evaluate Accuracy & Metrics
8. Predict Future Prices
9. Save Model

## Cell 1 - Install Dependencies

In [ ]:
# Install required libraries
!pip install -q yfinance pandas-ta scikit-learn keras tensorflow plotly

print('All packages installed!')

## Cell 2 - Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error,
    mean_absolute_percentage_error, r2_score
)
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Bidirectional,
    BatchNormalization, Input
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

import pandas_ta as ta

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {len(tf.config.list_physical_devices("GPU")) > 0}')

## Cell 3 - Download Stock Data (Yahoo Finance)

In [ ]:
# Configuration - change TARGET to predict a different stock
TICKERS    = ['AAPL', 'GOOG', 'MSFT', 'AMZN']
TARGET     = 'AAPL'   # stock to predict
START      = '2012-01-01'
END        = '2023-12-31'
SEQ_LEN    = 60       # look-back window in trading days
FORECAST   = 30       # days to forecast ahead
TEST_SPLIT = 0.15
VAL_SPLIT  = 0.10

print(f'Downloading {TICKERS} from {START} to {END} ...')
raw = yf.download(TICKERS, start=START, end=END, group_by='ticker', auto_adjust=True)

# Flatten MultiIndex columns
raw.columns = ['_'.join(col).strip() for col in raw.columns]
raw.index   = pd.to_datetime(raw.index)

print(f'Shape  : {raw.shape}')
print(f'Date range: {raw.index.min().date()} to {raw.index.max().date()}')
raw.head(3)

## Cell 4 - Data Cleaning & Quality Check

In [ ]:
# Extract target stock OHLCV
ohlcv_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
df = pd.DataFrame({col: raw[f'{TARGET}_{col}'] for col in ohlcv_cols})

print('=' * 50)
print(f'  TARGET STOCK : {TARGET}')
print('=' * 50)
print(f'Shape     : {df.shape}')
print(f'Date range: {df.index.min().date()} to {df.index.max().date()}')

# Step 1: Missing values
print(f'\nMissing values BEFORE cleaning:')
print(df.isnull().sum())
df = df.ffill().bfill()
print(f'\nMissing values AFTER cleaning:')
print(df.isnull().sum())

# Step 2: Duplicate dates
dupes = df.index.duplicated().sum()
print(f'\nDuplicate dates: {dupes}')
if dupes > 0:
    df = df[~df.index.duplicated(keep='last')]

# Step 3: Zero/negative prices
price_cols = ['Open', 'High', 'Low', 'Close']
bad_rows   = (df[price_cols] <= 0).any(axis=1).sum()
print(f'Zero/negative price rows: {bad_rows}')
if bad_rows > 0:
    df = df[(df[price_cols] > 0).all(axis=1)]

# Step 4: Outlier detection via IQR on daily returns
df['Daily_Return'] = df['Close'].pct_change()
Q1 = df['Daily_Return'].quantile(0.01)
Q3 = df['Daily_Return'].quantile(0.99)
outliers = ((df['Daily_Return'] < Q1) | (df['Daily_Return'] > Q3)).sum()
print(f'Extreme-return outliers (1-99th pct): {outliers}')
print('Keeping outliers - they are real market events LSTM should learn')

# Step 5: Sort chronologically
df = df.sort_index()

print(f'\nFinal clean shape: {df.shape}')
df.describe().round(2)

## Cell 5 - Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 14))

# Closing price
axes[0].plot(df.index, df['Close'], color='#00d4ff', lw=1.5, label='Close Price')
axes[0].fill_between(df.index, df['Close'], alpha=0.08, color='#00d4ff')
axes[0].set_title(f'{TARGET} Closing Price History', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price (USD)')
axes[0].legend()

# Volume
axes[1].bar(df.index, df['Volume'], color='#a855f7', alpha=0.6, label='Volume')
axes[1].set_title('Trading Volume', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Volume')
axes[1].legend()

# Daily return distribution
axes[2].hist(df['Daily_Return'].dropna(), bins=80, color='#f59e0b', edgecolor='none', alpha=0.8)
axes[2].axvline(0, color='red', ls='--', lw=1)
axes[2].set_title('Daily Return Distribution', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Daily Return')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

# Multi-stock normalized performance
fig, ax = plt.subplots(figsize=(16, 6))
colors = ['#00d4ff', '#f59e0b', '#10b981', '#f43f5e']
for ticker, color in zip(TICKERS, colors):
    close_col = f'{ticker}_Close'
    if close_col in raw.columns:
        series = raw[close_col].dropna()
        normalized = (series / series.iloc[0]) * 100
        ax.plot(normalized.index, normalized.values, color=color, lw=1.5, label=ticker)
ax.set_title('Normalised Stock Performance (Base = 100)', fontsize=14, fontweight='bold')
ax.set_ylabel('Normalised Price')
ax.legend()
plt.tight_layout()
plt.savefig('multi_stock.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlation heatmap
close_all   = pd.DataFrame({t: raw[f'{t}_Close'] for t in TICKERS}).dropna()
returns_all = close_all.pct_change().dropna()
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(returns_all.corr(), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax, linewidths=0.5, square=True)
ax.set_title('Return Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 6 - Feature Engineering (Technical Indicators)

In [ ]:
feat = df.copy()

# Trend indicators
feat['SMA_20']    = ta.sma(feat['Close'], length=20)
feat['SMA_50']    = ta.sma(feat['Close'], length=50)
feat['EMA_12']    = ta.ema(feat['Close'], length=12)
feat['EMA_26']    = ta.ema(feat['Close'], length=26)

# Momentum
feat['RSI_14']    = ta.rsi(feat['Close'], length=14)
macd              = ta.macd(feat['Close'], fast=12, slow=26, signal=9)
feat['MACD']      = macd['MACD_12_26_9']
feat['MACD_sig']  = macd['MACDs_12_26_9']
feat['MACD_hist'] = macd['MACDh_12_26_9']

# Volatility
bb                = ta.bbands(feat['Close'], length=20, std=2)
feat['BB_upper']  = bb['BBU_20_2.0']
feat['BB_lower']  = bb['BBL_20_2.0']
feat['BB_mid']    = bb['BBM_20_2.0']
feat['BB_width']  = (feat['BB_upper'] - feat['BB_lower']) / feat['BB_mid']
feat['ATR_14']    = ta.atr(feat['High'], feat['Low'], feat['Close'], length=14)

# Volume indicators
feat['OBV']       = ta.obv(feat['Close'], feat['Volume'])
feat['VWAP']      = (
    feat['Volume'] * (feat['High'] + feat['Low'] + feat['Close']) / 3
).cumsum() / feat['Volume'].cumsum()

# Price-derived features
feat['HL_ratio']     = feat['High'] / feat['Low']
feat['OC_ratio']     = feat['Open'] / feat['Close']
feat['Log_Return']   = np.log(feat['Close'] / feat['Close'].shift(1))
feat['Volatility_5'] = feat['Log_Return'].rolling(5).std()
feat['Volatility_20']= feat['Log_Return'].rolling(20).std()

# Lag features (previous 5 days close)
for lag in range(1, 6):
    feat[f'Close_lag{lag}'] = feat['Close'].shift(lag)

# Drop NaN rows from rolling windows
feat.dropna(inplace=True)

print(f'Feature set shape: {feat.shape}')
print(f'Total features ({len(feat.columns)}):')
print(list(feat.columns))

## Cell 7 - Prepare Sequences for LSTM

In [ ]:
FEATURE_COLS = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'SMA_20', 'SMA_50', 'EMA_12', 'EMA_26',
    'RSI_14', 'MACD', 'MACD_sig', 'MACD_hist',
    'BB_upper', 'BB_lower', 'BB_width', 'ATR_14',
    'OBV', 'VWAP',
    'HL_ratio', 'OC_ratio', 'Log_Return',
    'Volatility_5', 'Volatility_20',
    'Close_lag1', 'Close_lag2', 'Close_lag3', 'Close_lag4', 'Close_lag5'
]

TARGET_COL = 'Close'
data       = feat[FEATURE_COLS].values

# Time-series train/test split (NO SHUFFLE)
split_idx  = int(len(data) * (1 - TEST_SPLIT))
train_data = data[:split_idx]
test_data  = data[split_idx - SEQ_LEN:]   # include look-back context

print(f'Total  : {len(data)} samples')
print(f'Train  : {len(train_data)} samples')
print(f'Test   : {len(test_data)} samples')

# Scale with RobustScaler (handles outliers better than MinMax)
close_idx    = FEATURE_COLS.index(TARGET_COL)
scaler       = RobustScaler()
close_scaler = RobustScaler()

train_scaled = scaler.fit_transform(train_data)
test_scaled  = scaler.transform(test_data)
close_scaler.fit(train_data[:, close_idx].reshape(-1, 1))

# Build sliding window sequences
def make_sequences(data_scaled, seq_len, close_col_idx):
    X, y = [], []
    for i in range(seq_len, len(data_scaled)):
        X.append(data_scaled[i - seq_len:i])
        y.append(data_scaled[i, close_col_idx])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_train_full, y_train_full = make_sequences(train_scaled, SEQ_LEN, close_idx)
X_test, y_test             = make_sequences(test_scaled,  SEQ_LEN, close_idx)

# Validation split
val_idx  = int(len(X_train_full) * (1 - VAL_SPLIT))
X_train  = X_train_full[:val_idx]
y_train  = y_train_full[:val_idx]
X_val    = X_train_full[val_idx:]
y_val    = y_train_full[val_idx:]

print(f'\nX_train : {X_train.shape}')
print(f'y_train : {y_train.shape}')
print(f'X_val   : {X_val.shape}')
print(f'X_test  : {X_test.shape}')

## Cell 8 - Build the Stacked Bidirectional LSTM Model

In [ ]:
n_features = X_train.shape[2]

def build_lstm_model(seq_len, n_features, units=[256, 128, 64],
                     dropout=0.25, lr=1e-3, use_bidirectional=True):
    model = Sequential(name='LSTM_Stock_Predictor')
    model.add(Input(shape=(seq_len, n_features)))

    for i, unit in enumerate(units):
        return_seq = (i < len(units) - 1)
        if use_bidirectional:
            model.add(Bidirectional(
                LSTM(unit, return_sequences=return_seq, kernel_regularizer=l2(1e-4))
            ))
        else:
            model.add(LSTM(unit, return_sequences=return_seq, kernel_regularizer=l2(1e-4)))
        model.add(BatchNormalization())
        model.add(Dropout(dropout))

    model.add(Dense(64, activation='relu', kernel_regularizer=l2(1e-4)))
    model.add(Dropout(dropout / 2))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))

    model.compile(
        optimizer=Adam(learning_rate=lr, clipnorm=1.0),
        loss='huber',
        metrics=['mae']
    )
    return model

model = build_lstm_model(
    seq_len=SEQ_LEN,
    n_features=n_features,
    units=[256, 128, 64],
    dropout=0.25,
    lr=1e-3,
    use_bidirectional=True
)

model.summary()

## Cell 9 - Train the Model

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=8,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        'best_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    )
]

EPOCHS     = 150
BATCH_SIZE = 64

print(f'Training LSTM - {EPOCHS} epochs max (EarlyStopping active)...')

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    shuffle=False,
    verbose=1
)

print(f'\nTraining complete. Best val_loss: {min(history.history["val_loss"]):.6f}')

## Cell 10 - Plot Training & Validation Loss

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(history.history['loss'],     label='Train Loss',  color='#00d4ff', lw=2)
axes[0].plot(history.history['val_loss'], label='Val Loss',    color='#f43f5e', lw=2)
axes[0].set_title('Model Loss (Huber)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].set_yscale('log')

axes[1].plot(history.history['mae'],     label='Train MAE',  color='#10b981', lw=2)
axes[1].plot(history.history['val_mae'], label='Val MAE',    color='#f59e0b', lw=2)
axes[1].set_title('Model MAE', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 11 - Evaluate on Test Set (Full Metrics)

In [ ]:
y_pred_scaled = model.predict(X_test, verbose=0).flatten()

# Inverse transform to real USD prices
y_pred_real = close_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_test_real = close_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

# Metrics
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
mae  = mean_absolute_error(y_test_real, y_pred_real)
mape = mean_absolute_percentage_error(y_test_real, y_pred_real) * 100
r2   = r2_score(y_test_real, y_pred_real)
da   = np.mean(np.sign(np.diff(y_test_real)) == np.sign(np.diff(y_pred_real))) * 100

print('=' * 50)
print(f'  EVALUATION METRICS - {TARGET}')
print('=' * 50)
print(f'  RMSE                 : ${rmse:.4f}')
print(f'  MAE                  : ${mae:.4f}')
print(f'  MAPE                 : {mape:.2f} %')
print(f'  R-Squared            : {r2:.4f}')
print(f'  Directional Accuracy : {da:.2f} %')
print('=' * 50)

# Actual vs Predicted
n_test = len(y_test_real)
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(range(n_test), y_test_real, label='Actual Price',    color='#00d4ff', lw=2)
ax.plot(range(n_test), y_pred_real, label='Predicted Price', color='#f43f5e', lw=2, ls='--')
ax.fill_between(range(n_test), y_test_real, y_pred_real, alpha=0.1, color='#f59e0b')
ax.set_title(f'{TARGET} - Actual vs Predicted (Test Set)\n'
             f'R2={r2:.4f}  MAPE={mape:.2f}%  Directional Acc={da:.1f}%',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Test Day')
ax.set_ylabel('Price (USD)')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('test_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

# Residuals
residuals = y_test_real - y_pred_real
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].scatter(y_pred_real, residuals, alpha=0.4, color='#a855f7', s=15)
axes[0].axhline(0, color='red', lw=1)
axes[0].set_title('Residuals vs Fitted', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residual')
axes[1].hist(residuals, bins=40, color='#10b981', edgecolor='none', alpha=0.8)
axes[1].axvline(0, color='red', lw=1)
axes[1].set_title('Residual Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Residual ($)')
plt.tight_layout()
plt.savefig('residuals.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 12 - Forecast Next N Days

In [ ]:
# Recursive multi-step forecast
last_sequence    = test_scaled[-SEQ_LEN:]
forecast_scaled  = []
current_seq      = last_sequence.copy()

for _ in range(FORECAST):
    pred = model.predict(current_seq[np.newaxis], verbose=0)[0, 0]
    forecast_scaled.append(pred)
    new_row            = current_seq[-1].copy()
    new_row[close_idx] = pred
    current_seq        = np.vstack([current_seq[1:], new_row])

forecast_real = close_scaler.inverse_transform(
    np.array(forecast_scaled).reshape(-1, 1)
).flatten()

last_date    = feat.index[-1]
future_dates = pd.bdate_range(start=last_date + pd.Timedelta(days=1), periods=FORECAST)

# Plot
history_window = 120
hist_close     = feat['Close'].iloc[-history_window:]

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(hist_close.index, hist_close.values,
        color='#00d4ff', lw=2, label='Historical Close')
ax.plot(future_dates, forecast_real,
        color='#f43f5e', lw=2.5, ls='--', marker='o', ms=5,
        label=f'Forecast (+{FORECAST} days)')
ax.fill_between(future_dates,
                forecast_real * 0.97, forecast_real * 1.03,
                alpha=0.15, color='#f43f5e', label='3% confidence band')
ax.axvline(x=feat.index[-1], color='#f59e0b', ls=':', lw=1.5, label='Forecast Start')
ax.set_title(f'{TARGET} - {FORECAST}-Day Price Forecast', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('forecast.png', dpi=150, bbox_inches='tight')
plt.show()

forecast_df = pd.DataFrame({'Date': future_dates, f'{TARGET}_Forecast': forecast_real.round(2)})
forecast_df.set_index('Date', inplace=True)
print(f'\n{FORECAST}-Day Price Forecast for {TARGET}:')
print(forecast_df.to_string())

## Cell 13 - (Optional) Hyperparameter Tuning with Keras Tuner

In [ ]:
# Set RUN_TUNER = True to enable (takes 30-60 min on GPU)
RUN_TUNER = False

if RUN_TUNER:
    !pip install -q keras-tuner
    import keras_tuner as kt

    def model_builder(hp):
        units1  = hp.Choice('units1',  [64, 128, 256])
        units2  = hp.Choice('units2',  [32, 64, 128])
        dropout = hp.Float('dropout',  0.1, 0.4, step=0.05)
        lr      = hp.Choice('lr',      [1e-2, 1e-3, 5e-4, 1e-4])
        bidir   = hp.Boolean('bidir')

        m = Sequential()
        m.add(Input(shape=(SEQ_LEN, n_features)))
        for unit, ret_seq in zip([units1, units2], [True, False]):
            layer = LSTM(unit, return_sequences=ret_seq)
            m.add(Bidirectional(layer) if bidir else layer)
            m.add(BatchNormalization())
            m.add(Dropout(dropout))
        m.add(Dense(32, activation='relu'))
        m.add(Dense(1))
        m.compile(optimizer=Adam(lr), loss='huber', metrics=['mae'])
        return m

    tuner = kt.BayesianOptimization(
        model_builder,
        objective='val_loss',
        max_trials=20,
        directory='kt_dir',
        project_name='stock_lstm'
    )
    tuner.search(
        X_train, y_train,
        epochs=30,
        validation_data=(X_val, y_val),
        callbacks=[EarlyStopping(patience=5)],
        verbose=1
    )
    best_hp = tuner.get_best_hyperparameters(1)[0]
    print('Best hyperparameters:')
    for key in ['units1', 'units2', 'dropout', 'lr', 'bidir']:
        print(f'   {key}: {best_hp.get(key)}')
    tuner.get_best_models(1)[0].save('best_tuned_model.keras')
else:
    print('Tuner skipped. Set RUN_TUNER = True to enable.')

## Cell 14 - Save Everything

In [ ]:
import joblib
import json

model.save('stock_lstm_model.keras')
print('Model saved -> stock_lstm_model.keras')

joblib.dump(scaler,       'feature_scaler.pkl')
joblib.dump(close_scaler, 'close_scaler.pkl')
print('Scalers saved -> feature_scaler.pkl, close_scaler.pkl')

feat.to_csv('stock_features_clean.csv')
print('Dataset saved -> stock_features_clean.csv')

forecast_df.to_csv('forecast.csv')
print('Forecast saved -> forecast.csv')

metrics = {
    'ticker': TARGET,
    'rmse':   round(float(rmse), 4),
    'mae':    round(float(mae),  4),
    'mape':   round(float(mape), 4),
    'r2':     round(float(r2),   4),
    'directional_accuracy': round(float(da), 2)
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved -> metrics.json')
print(f'\nSummary: {json.dumps(metrics, indent=2)}')

# Auto-download files if running in Colab
try:
    from google.colab import files
    for fname in ['stock_lstm_model.keras', 'feature_scaler.pkl',
                  'close_scaler.pkl', 'stock_features_clean.csv',
                  'forecast.csv', 'metrics.json']:
        files.download(fname)
    print('\nAll files downloaded!')
except ImportError:
    print('Not in Colab - files saved locally.')

## Cell 15 - Quick Sanity Check (Inference on 1 Example)

In [ ]:
loaded_model  = load_model('stock_lstm_model.keras')
loaded_scaler = joblib.load('feature_scaler.pkl')
loaded_close  = joblib.load('close_scaler.pkl')

sample_raw    = feat[FEATURE_COLS].values[-SEQ_LEN:]
sample_scaled = loaded_scaler.transform(sample_raw)
sample_input  = sample_scaled[np.newaxis]

pred_scaled   = loaded_model.predict(sample_input, verbose=0)[0, 0]
pred_price    = loaded_close.inverse_transform([[pred_scaled]])[0, 0]

last_actual   = feat['Close'].iloc[-1]
change_pct    = (pred_price - last_actual) / last_actual * 100

print(f'Next trading day prediction for {TARGET}:')
print(f'   Last known Close : ${last_actual:.2f}')
print(f'   Predicted Close  : ${pred_price:.2f}')
print(f'   Expected change  : {change_pct:+.2f} %')
print(f'   Direction        : {"UP" if change_pct > 0 else "DOWN"}')